# Benchmarking Machine Learning Classifiers for Heart Disease Prediction

| Field | Detail |
|---|---|
| **Course** | PMIT_6121: Machine Learning |
| **Assignment Title** | Comparative Analysis of SVM, KNN, and Naive Bayes Classifiers |
| **Dataset** | CDC Personal Key Indicators of Heart Disease (Kaggle) |
| **Student Name** | [Md. Abu Sufiyan Rakib] |
| **Student ID** | [252001] |
| **Due Date** | 15 June 2026 |

---

## Notebook Structure

| Section | Content |
|---|---|
| **0** | Environment setup & imports |
| **1** | Load & inspect dataset |
| **2** | Exploratory Data Analysis (EDA) |
| **3** | Preprocessing pipeline |
| **4** | Model training & tuning |
| **5** | Performance evaluation |
| **6** | Results summary |

**Dataset download:**
`kaggle datasets download -d kamilpytlak/personal-key-indicators-of-heart-disease`
Unzip and place `heart_2020_cleaned.csv` in the same folder as this notebook.

---
## Section 0 — Environment Setup & Imports

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.svm             import SVC
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.naive_bayes     import GaussianNB
from sklearn.metrics         import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
)

DARK_BLUE = '#1F3864'; MID_BLUE = '#2E75B6'
ACCENT    = '#C00000'; GOLD     = '#F39C12'

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'axes.spines.top': False,
    'axes.spines.right': False,   'axes.facecolor': '#F5F5F5',
    'figure.facecolor': 'white',  'axes.grid': True,
    'grid.alpha': 0.4,            'grid.color': 'white',
})

FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)
print('Environment ready. Figures will be saved to:', FIG_DIR)

---
## Section 1 — Load & Inspect Dataset

In [ ]:
CSV = 'heart_2020_cleaned.csv'
df  = pd.read_csv(CSV)

print('=' * 55)
print('DATASET SUMMARY')
print('=' * 55)
print(f'  Rows      : {df.shape[0]:,}')
print(f'  Columns   : {df.shape[1]}')
print(f'  Missing   : {df.isnull().sum().sum()}')
print(f'  Duplicates: {df.duplicated().sum():,}')
print()
vc = df['HeartDisease'].value_counts()
for label, cnt in vc.items():
    print(f'  {label:5s} -> {cnt:>7,}  ({cnt/len(df)*100:.1f}%)')

df.head()

In [ ]:
print('Column dtypes:')
print(df.dtypes.to_string())
print()
df.describe().round(2)

---
## Section 2 — Exploratory Data Analysis (EDA)

In [ ]:
# Figure 1: Class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle('Figure 1 - Target Class Distribution', fontsize=13, fontweight='bold', color=DARK_BLUE)

counts = df['HeartDisease'].value_counts().sort_index()
labels = ['No Heart Disease', 'Heart Disease']
colors = [MID_BLUE, ACCENT]

wedges, texts, autotexts = ax1.pie(
    counts, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2.5},
    textprops={'fontsize': 10})
for at in autotexts:
    at.set_color('white'); at.set_fontweight('bold')
ax1.set_title('Proportion', fontsize=11, color=DARK_BLUE)

bars = ax2.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.55)
for bar, val in zip(bars, counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height()+1500,
             f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold', color=DARK_BLUE)
ax2.set_ylabel('Number of Records')
ax2.set_title('Record Counts', fontsize=11, color=DARK_BLUE)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observation: ~91% negative class - raw accuracy is misleading. Use Recall & F1.')

In [ ]:
# Figure 2: BMI distribution by class
fig, ax = plt.subplots(figsize=(10, 4.5))
fig.suptitle('Figure 2 - BMI Distribution by Heart Disease Status', fontsize=13, fontweight='bold', color=DARK_BLUE)

bmi_no  = df[df['HeartDisease']=='No' ]['BMI'].clip(10, 60)
bmi_yes = df[df['HeartDisease']=='Yes']['BMI'].clip(10, 60)

ax.hist(bmi_no,  bins=60, alpha=0.65, color=MID_BLUE, label='No Heart Disease', density=True)
ax.hist(bmi_yes, bins=60, alpha=0.65, color=ACCENT,   label='Heart Disease',    density=True)
ax.axvline(bmi_no.mean(),  color=MID_BLUE, linestyle='--', linewidth=2, label=f'Mean (No HD) = {bmi_no.mean():.2f}')
ax.axvline(bmi_yes.mean(), color=ACCENT,   linestyle='--', linewidth=2, label=f'Mean (HD)    = {bmi_yes.mean():.2f}')
ax.set_xlabel('BMI'); ax.set_ylabel('Density'); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig2_bmi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Heart disease rate by age category
AGE_ORDER = ['18-24','25-29','30-34','35-39','40-44','45-49',
             '50-54','55-59','60-64','65-69','70-74','75-79','80 or older']

df_tmp = df.copy()
df_tmp['HD_bin'] = (df_tmp['HeartDisease']=='Yes').astype(int)
age_hd = df_tmp.groupby('AgeCategory')['HD_bin'].mean()*100
age_hd = age_hd.reindex(AGE_ORDER)

fig, ax = plt.subplots(figsize=(12, 4.5))
fig.suptitle('Figure 3 - Heart Disease Prevalence by Age Category', fontsize=13, fontweight='bold', color=DARK_BLUE)

bar_cols = [ACCENT if v > 10 else MID_BLUE for v in age_hd.values]
bars = ax.bar(age_hd.index, age_hd.values, color=bar_cols, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, age_hd.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xlabel('Age Category'); ax.set_ylabel('Heart Disease Prevalence (%)')
ax.tick_params(axis='x', rotation=35)
ax.legend(handles=[
    mpatches.Patch(color=MID_BLUE, label='Prevalence <= 10%'),
    mpatches.Patch(color=ACCENT,   label='Prevalence > 10%'),
], fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig3_age_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Risk factor comparison
no_hd  = df[df['HeartDisease']=='No']
yes_hd = df[df['HeartDisease']=='Yes']

factors  = ['Smoking', 'Diabetic', 'Physical\nInactivity']
no_vals  = [(no_hd['Smoking']=='Yes').mean()*100,
            (no_hd['Diabetic']=='Yes').mean()*100,
            (no_hd['PhysicalActivity']=='No').mean()*100]
yes_vals = [(yes_hd['Smoking']=='Yes').mean()*100,
            (yes_hd['Diabetic']=='Yes').mean()*100,
            (yes_hd['PhysicalActivity']=='No').mean()*100]

fig, ax = plt.subplots(figsize=(9, 4.5))
fig.suptitle('Figure 4 - Risk Factor Prevalence by Class', fontsize=13, fontweight='bold', color=DARK_BLUE)
x = np.arange(len(factors)); w = 0.35
b1 = ax.bar(x-w/2, no_vals,  w, label='No Heart Disease', color=MID_BLUE, edgecolor='white', linewidth=1.5)
b2 = ax.bar(x+w/2, yes_vals, w, label='Heart Disease',    color=ACCENT,   edgecolor='white', linewidth=1.5)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(factors)
ax.set_ylabel('Prevalence (%)'); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig4_risk_factors.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Preprocessing Pipeline

Steps:
1. **Label-encode target** — `'Yes'` -> `1`, `'No'` -> `0`
2. **One-hot encode** all categorical columns (`drop_first=True` avoids dummy-variable trap)
3. **80/20 stratified split** — preserves class ratio in both subsets
4. **StandardScaler** — fit on train only, transform both sets (no data leakage)

> **Why is scaling essential for SVM and KNN but optional for Naive Bayes?**
> - **SVM**: maximises margin via distance/dot-product — unscaled features with large ranges dominate the hyperplane.
> - **KNN**: uses Euclidean distance — a feature with range x1000 overwhelms a binary 0/1 feature.
> - **Gaussian NB**: estimates per-feature Gaussians independently. Scaling re-parameterises each Gaussian but leaves posterior probabilities identical — mathematically irrelevant.

In [ ]:
# Step 1: Encode target
df2 = df.copy()
df2['HeartDisease'] = df2['HeartDisease'].map({'Yes': 1, 'No': 0})

# Step 2: One-hot encode categorical columns
cat_cols = df2.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode ({len(cat_cols)}): {cat_cols}')
df2 = pd.get_dummies(df2, columns=cat_cols, drop_first=True)
print(f'Total features after encoding: {df2.shape[1]-1}')

# Step 3: Train/test split
X = df2.drop('HeartDisease', axis=1)
y = df2['HeartDisease']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}  |  Features: {X_train.shape[1]}')

# Step 4: StandardScaler
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform on train only
X_test_s  = scaler.transform(X_test)         # transform only on test
print(f'Post-scaling mean={X_train_s.mean():.4f}  std={X_train_s.std():.4f}')

---
## Section 4 — Model Training & Hyperparameter Tuning

In [ ]:
# 4.1 Support Vector Machine
# Testing: kernel in {linear, rbf}, C=1.0
print('Training SVM (linear) ...')
svm_linear = SVC(kernel='linear', C=1.0, random_state=42)
svm_linear.fit(X_train_s, y_train)
y_svm_linear = svm_linear.predict(X_test_s)

print('Training SVM (rbf) ...')
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_rbf.fit(X_train_s, y_train)
y_svm_rbf = svm_rbf.predict(X_test_s)

print('\n--- SVM (linear) ---')
print(classification_report(y_test, y_svm_linear, target_names=['No HD','Heart Disease']))
print('--- SVM (RBF) ---')
print(classification_report(y_test, y_svm_rbf,    target_names=['No HD','Heart Disease']))

In [ ]:
# 4.2 K-Nearest Neighbors
# Testing: K in {3, 5, 11}
knn_results = {}
for k in [3, 5, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    y_pred = knn.predict(X_test_s)
    knn_results[k] = {
        'preds'   : y_pred,
        'accuracy': accuracy_score(y_test, y_pred),
        'recall'  : recall_score(y_test, y_pred),
        'f1'      : f1_score(y_test, y_pred),
    }
    print(f'K={k:2d}  Accuracy={knn_results[k]["accuracy"]:.4f}  Recall={knn_results[k]["recall"]:.4f}  F1={knn_results[k]["f1"]:.4f}')

best_k     = max(knn_results, key=lambda k: knn_results[k]['f1'])
y_knn_best = knn_results[best_k]['preds']
print(f'\nBest K = {best_k} (by F1-Score)')
print(classification_report(y_test, y_knn_best, target_names=['No HD','Heart Disease']))

In [ ]:
# 4.3 Gaussian Naive Bayes (default hyperparameters)
gnb   = GaussianNB()
gnb.fit(X_train_s, y_train)
y_gnb = gnb.predict(X_test_s)

print('--- Gaussian Naive Bayes (default) ---')
print(classification_report(y_test, y_gnb, target_names=['No HD','Heart Disease']))

---
## Section 5 — Performance Evaluation

In [ ]:
# Figure 5: KNN K-value comparison
k_vals = [3, 5, 11]
acc_k  = [knn_results[k]['accuracy'] for k in k_vals]
rec_k  = [knn_results[k]['recall']   for k in k_vals]
f1_k   = [knn_results[k]['f1']       for k in k_vals]

fig, ax = plt.subplots(figsize=(8, 4.5))
fig.suptitle('Figure 5 - KNN Performance Across K Values', fontsize=13, fontweight='bold', color=DARK_BLUE)
x = np.arange(len(k_vals)); w = 0.25
ax.bar(x-w, acc_k, w, label='Accuracy', color=MID_BLUE, edgecolor='white')
ax.bar(x,   rec_k, w, label='Recall',   color=ACCENT,   edgecolor='white')
ax.bar(x+w, f1_k,  w, label='F1-Score', color=GOLD,     edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels([f'K = {k}' for k in k_vals])
ax.set_ylabel('Score'); ax.set_ylim(0, 1.1); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig5_knn_k_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Cross-model comparison
model_names    = ['SVM\nLinear','SVM\nRBF',f'KNN\nK={best_k}','Naive\nBayes']
all_preds      = [y_svm_linear, y_svm_rbf, y_knn_best, y_gnb]
bar_colors     = [MID_BLUE, MID_BLUE, GOLD, ACCENT]
metrics_to_plot = {
    'Accuracy' : [accuracy_score(y_test, p)  for p in all_preds],
    'Precision': [precision_score(y_test, p) for p in all_preds],
    'Recall'   : [recall_score(y_test, p)    for p in all_preds],
    'F1-Score' : [f1_score(y_test, p)        for p in all_preds],
}
fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
fig.suptitle('Figure 6 - Comparative Model Performance', fontsize=13, fontweight='bold', color=DARK_BLUE)
for ax, (metric, vals) in zip(axes, metrics_to_plot.items()):
    bars = ax.bar(model_names, vals, color=bar_colors, edgecolor='white', linewidth=1.5)
    bars[int(np.argmax(vals))].set_edgecolor('#FF0000'); bars[int(np.argmax(vals))].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_title(metric, fontsize=11, fontweight='bold', color=DARK_BLUE)
    ax.set_ylim(0, min(1.15, max(vals)*1.25))
    ax.tick_params(axis='x', labelsize=8)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig6_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 7: Confusion matrices
models_cm = [
    ('SVM (Linear)',      y_svm_linear),
    ('SVM (RBF)',         y_svm_rbf),
    (f'KNN (K={best_k})', y_knn_best),
    ('Naive Bayes',       y_gnb),
]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
fig.suptitle('Figure 7 - Confusion Matrices for All Classifiers',
             fontsize=13, fontweight='bold', color=DARK_BLUE, y=1.02)
for ax, (title, y_pred) in zip(axes, models_cm):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['No HD','HD'], yticklabels=['No HD','HD'],
                linewidths=0.5, linecolor='white', annot_kws={'size':11,'weight':'bold'})
    ax.set_title(title, fontsize=11, fontweight='bold', color=DARK_BLUE, pad=10)
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual',    fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/fig7_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6 — Results Summary

In [ ]:
def get_metrics(y_true, y_pred, label):
    return {
        'Model'    : label,
        'Accuracy' : round(accuracy_score(y_true, y_pred),  4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall'   : round(recall_score(y_true, y_pred),    4),
        'F1-Score' : round(f1_score(y_true, y_pred),        4),
    }

summary = pd.DataFrame([
    get_metrics(y_test, y_svm_linear,                'SVM  (kernel=linear, C=1.0)'),
    get_metrics(y_test, y_svm_rbf,                   'SVM  (kernel=rbf, C=1.0)'),
    get_metrics(y_test, knn_results[3]['preds'],     'KNN  (K=3)'),
    get_metrics(y_test, knn_results[5]['preds'],     'KNN  (K=5)'),
    get_metrics(y_test, knn_results[11]['preds'],    'KNN  (K=11)'),
    get_metrics(y_test, y_gnb,                       'Naive Bayes (Gaussian, default)'),
])

best_idx = summary['Recall'].idxmax()
print('='*70)
print('FINAL PERFORMANCE SUMMARY  (positive class: HeartDisease=Yes)')
print('='*70)
print(summary.to_string(index=False))
print()
print(f'Best Recall -> {summary.loc[best_idx, "Model"]}  (Recall={summary.loc[best_idx, "Recall"]})')
print('Clinical recommendation: Gaussian Naive Bayes is safest for hospital screening.')
summary

---
## Discussion Notes

### False Negative vs False Positive — Clinical Priority

| Error | Consequence | Priority |
|---|---|---|
| **False Negative** | Heart disease missed; patient untreated -> potential fatal event | **Eliminate first** |
| **False Positive** | Healthy patient over-investigated; cost & anxiety | Acceptable trade-off |

**Conclusion:** Recall (Sensitivity) must be the primary optimisation target.

### Future Improvements
1. **SMOTE** — Synthetic Minority Oversampling to address the 91:9 class imbalance.
2. **Ensemble methods** — XGBoost / LightGBM with class-weight tuning.
3. **Decision threshold tuning** — Lower threshold (e.g. P > 0.25) directly increases Recall.